# 🚀 Crypto-Pulse: Sentiment Intelligence Dashboard
### Milestone 3: Real-time FinBERT Insights

This notebook explores the relationship between market news sentiment and cryptocurrency price movements using the FinBERT model.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv('../.env')

# Database connection
DB_URL = f"postgresql://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}@{os.getenv('POSTGRES_HOST', 'localhost')}:5432/{os.getenv('POSTGRES_DB')}"
engine = create_engine(DB_URL)

print("✅ Dashboard Ready to Load Data")

ModuleNotFoundError: No module named 'plotly'

## 1. Overall Market Sentiment Distribution
How is the market feeling today? We categorize news into **Positive**, **Negative**, and **Neutral** using FinBERT.

In [ ]:
query = "SELECT sentiment_label, count(*) as count FROM silver.news_sentiment GROUP BY sentiment_label"
df_sentiment = pd.read_sql(query, engine)

fig = px.pie(df_sentiment, values='count', names='sentiment_label', 
             title='Global News Sentiment Distribution',
             color_discrete_map={'positive':'#00cc96', 'negative':'#ef553b', 'neutral':'#636efa'})
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

NameError: name 'engine' is not defined

## 2. Sentiment Heatmap per Asset
Which assets are currently experiencing the most 'Positive' or 'Negative' coverage?

In [ ]:
query = """
SELECT 
    symbol, 
    sentiment_label, 
    count(*) as count 
FROM silver.news_sentiment 
GROUP BY symbol, sentiment_label
"""
df_asset_sentiment = pd.read_sql(query, engine)

fig = px.bar(df_asset_sentiment, x="symbol", y="count", color="sentiment_label", 
             title="Asset-Specific Sentiment Analysis",
             barmode="group",
             color_discrete_map={'positive':'#00cc96', 'negative':'#ef553b', 'neutral':'#636efa'})
fig.show()

## 3. High Impact News Items
The titles below were identified as having the strongest sentiment scores.

In [ ]:
query = """
SELECT 
    title, 
    sentiment_label, 
    ROUND(sentiment_score::numeric, 4) as score, 
    published_at 
FROM silver.news_sentiment 
ORDER BY ABS(sentiment_score) DESC 
LIMIT 10
"""
df_top_news = pd.read_sql(query, engine)
df_top_news.style.background_gradient(subset=['score'], cmap='RdYlGn')

## 4. Sentiment Momentum (Timeline)
Are we becoming more optimistic or pessimistic over the last few hours?

In [ ]:
query = """
SELECT 
    date_trunc('hour', published_at) as hour, 
    AVG(sentiment_score) as avg_sentiment
FROM silver.news_sentiment 
GROUP BY hour 
ORDER BY hour
"""
df_timeline = pd.read_sql(query, engine)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_timeline['hour'], y=df_timeline['avg_sentiment'], 
                         mode='lines+markers', name='Avg Sentiment'))
fig.update_layout(title='Sentiment Momentum Timeline', 
                  xaxis_title='Time', yaxis_title='Avg Score (-1 to 1)')
fig.show()